In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import seaborn as sns

# Carregar dataset público Lending Club Loan Data (Kaggle)
arquivo = '../data/raw/accepted_2007_to_2018Q4.csv'

# Sorteio para ler apenas 5% da base de dados e economizar memória RAM
percentual = 0.05
skip_logic = lambda i: i > 0 and np.random.rand() > percentual
print("Carregando amostra de dados...")
df = pd.read_csv(arquivo, skiprows=skip_logic, low_memory=False)

# Exploração dos dados
print(df.info())
print(df.describe())

Carregando amostra de dados
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 113681 entries, 0 to 113680
Columns: 151 entries, id to settlement_term
dtypes: float64(113), object(38)
memory usage: 131.0+ MB
None
       member_id      loan_amnt    funded_amnt  funded_amnt_inv  \
count        0.0  113680.000000  113680.000000    113680.000000   
mean         NaN   15046.419775   15040.218596     15021.811195   
std          NaN    9189.010903    9187.363819      9190.712023   
min          NaN     500.000000     500.000000         0.000000   
25%          NaN    8000.000000    8000.000000      8000.000000   
50%          NaN   12950.000000   12900.000000     12800.000000   
75%          NaN   20000.000000   20000.000000     20000.000000   
max          NaN   40000.000000   40000.000000     40000.000000   

            int_rate    installment    annual_inc            dti  \
count  113680.000000  113680.000000  1.136800e+05  113577.000000   
mean       13.107487     445.930908  7.782326e+0

In [16]:
print("Iniciando a limpeza dos dados...")
# ==========================================
# PASSO 1: Remover colunas irrelevantes
# ==========================================
limite_minimo_preenchido = len(df) * 0.70 # Manter apenas colunas com pelo menos 70% de preenchimento
df_limpo = df.dropna(thresh=limite_minimo_preenchido, axis=1)

# ==========================================
# PASSO 2: Filtar o status do empréstimo (A variável alvo)
# ==========================================
status_validos = ['Fully Paid', 'Charged Off']
df_limpo = df_limpo[df_limpo['loan_status'].isin(status_validos)].copy() # A coluna 'loan_status' descreve o que aconteceu com o empréstimo

# ==========================================
# PASSO 3: Criar a coluna inadimplente (0 ou 1)
# ==========================================
df_limpo['inadimplente'] = (df_limpo['loan_status'] == 'Charged Off').astype(int)

# ==========================================
# RESULTADOS DA LIMPEZA
# ==========================================
print("\n--- Resultados da Limpeza Inicial ---")
print(f"Colunas originais: {df.shape[1]} -> Sobraram: {df_limpo.shape[1]} colunas úteis.")
print(f"Linhas originais: {df.shape[0]} -> Sobraram: {df_limpo.shape[0]} empréstimos finalizados.")

# Quantos pagaram vs quantos deram calote
print("\n Distribuição de Inadimplência:")
print(df_limpo['inadimplente'].value_counts(normalize=True) * 100)

Iniciando a limpeza dos dados...

--- Resultados da Limpeza Inicial ---
Colunas originais: 151 -> Sobraram: 94 colunas úteis.
Linhas originais: 113681 -> Sobraram: 67839 empréstimos finalizados.

 Distribuição de Inadimplência:
inadimplente
0    79.946638
1    20.053362
Name: proportion, dtype: float64


In [17]:
print(f"Linhas antes de tratar os nulos: {df_limpo.shape[0]}")

# 1. Separar as colunas numéricas e categóricas (texto)
colunas_numericas = df_limpo.select_dtypes(include=['float64', 'int64']).columns
colunas_categoricas = df_limpo.select_dtypes(include=['object']).columns

# 2. Preencher os nulos das colunas numéricas com a mediana
for col in colunas_numericas:
    if df_limpo[col].isnull().sum() > 0:
        mediana = df_limpo[col].median()
        df_limpo[col] = df_limpo[col].fillna(mediana)

# 3. Preencher os nulos das colunas categóricas com a string "Missing" (Desconhecido)
for col in colunas_categoricas:
    if df_limpo[col].isnull().sum() > 0:
        df_limpo[col] = df_limpo[col].fillna("Missing")

# 4. Verificação final
nulos_restantes = df_limpo.isnull().sum().sum()
print(f"Total de valores nulos restantes no DataFrame: {nulos_restantes}")
print(f"Dimensões finais do dataset limpo e tratado: {df_limpo.shape}")

Linhas antes de tratar os nulos: 67839
Total de valores nulos restantes no DataFrame: 0
Dimensões finais do dataset limpo e tratado: (67839, 94)


In [18]:
# Salvar o dataset limpo em um novo arquivo CSV
df_limpo.to_csv('../data/processed/data_cleaned.csv', index=False)
print("Dados tratados salvos com sucesso!")

Dados tratados salvos com sucesso!
